# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment (Python 3.8 or newer), on Windows, macOS and Linux.

The watermark engine is [audiowmark](https://github.com/swesterfeld/audiowmark), a compiled program that this notebook runs through **Docker**, like the Docker route in the Jemula setup:
- **Windows, macOS:** install and start [Docker Desktop](https://docs.docker.com/get-docker/).
- **Linux:** install Docker Engine, or build audiowmark yourself (see `Audio/docker/Dockerfile`); a patched `audiowmark` on your `PATH` is used automatically.

The first run compiles the engine once: a few minutes, and it needs internet. **Run the setup cell before the lab session**, not during it.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install --quiet -r Audio/requirements-audio.txt

### 1. Practical Over-The-Air Audio Communication

You will hide a short text message inside a music clip, play the clip through a speaker, record it with a microphone, and read the message back out of the recording.

The watermark is designed to be inaudible. The one setting that matters is the **level**:

| level | name | audio frames per message bit |
|---|---|---|
| 1 | very robust | 18 |
| 2 | robust | 12 |
| 3 | balanced | 6 |
| 4 | fast | 5 |
| 5 | faster | 4 |
| 6 | maximum | 2 |

Every level adds the same amount of watermark energy, so they all sound alike. What changes is how much audio protects each bit. Fewer frames per bit leave room for more characters but give each bit less protection: **more data means less robustness.** Finding where that trade-off breaks for your speaker and microphone is the point of this lab.

#### 1.1 Setup

In [ ]:
# --- Imports: libraries and functions used in this notebook ---
import os
from IPython.display import Audio, display
from Audio.audio_watermark import (AudioWatermark, setup, show_levels, simulate_channel, SCENARIOS,
                                   check_audio_loop, play_and_record, list_audio_devices, log_measurement)

# --- Check everything; the first run compiles the watermark engine (a few minutes) ---
READY = setup()
assert READY, "Setup is not complete: fix the [!!] lines above, then run this cell again."

**If setup is not ready**, the `[!!]` line says what to do. The usual ones:
- *Docker is not installed / not running*: install or start Docker Desktop and wait until it reports that it is running. On Windows, Docker Desktop needs WSL 2, as for the Jemula Docker route.
- *This user may not use Docker* (Linux): `sudo usermod -aG docker $USER`, then log out and back in.
- *Compiling failed with a network error*: the first run downloads the audiowmark source; connect to the internet and run the cell again.
- *Docker cannot read the work files*: if your user folder name contains unusual characters, run `os.environ["AUDIO_WATERMARK_TMP"] = "C:/wmtmp"` (any simple folder) in a cell, then this cell again.

`[--]` lines are optional features and never block the lab.

#### 1.2 Inputs

Any audio file works: its sample rate, channels and bit depth are kept in the output. Your message can use letters, digits and punctuation; each character costs 8 bits.

In [ ]:
# INPUT
INPUT_AUDIO_FILE  = "Audio/CrabRave.wav"      # <-- Change this. The music that carries your message
OUTPUT_AUDIO_FILE = "CrabRave_Encoded.wav"    # <-- Change this. The file you will play over the air
MESSAGE           = "Hello WNMC"               # <-- Change this. The message to transmit
LEVEL             = 2                          # <-- Change this. 1 (most robust) ... 6 (most data)
STRENGTH          = 15                         # <-- Watermark loudness: 15 is inaudible, higher is more robust

#### 1.3 How Much Fits?

The level sets a ceiling: how many characters fit into this clip. What costs robustness is how much of that ceiling you use. A message shorter than the maximum is repeated more often inside the clip, and every repeat adds protection. So a short message at level 5 can be as robust as the same message at level 2. The levels only differ once you fill them.

In [ ]:
show_levels(INPUT_AUDIO_FILE, MESSAGE)

#### 1.4 Embedding the Message

In [ ]:
wm = AudioWatermark(level=LEVEL, strength=STRENGTH)
sent = wm.embed(INPUT_AUDIO_FILE, OUTPUT_AUDIO_FILE, MESSAGE)
print(sent)

Listen to both versions. Can you hear a difference?

In [ ]:
print("Original:"); display(Audio(INPUT_AUDIO_FILE))
print("Encoded:");  display(Audio(OUTPUT_AUDIO_FILE))

#### 1.5 Transmitting Over the Air

Play `CrabRave_Encoded.wav` on a speaker and record it with a microphone at a measured distance. Save the recording next to this notebook as `CrabRave_Recorded.wav`. `.m4a` and `.mp3` recordings work too; just set the file name in 1.6.

**Setup Recommendations**
This section is continuously updated by usage reports.

**Phone (as Microphone)**
- Any recorder app works. iPhone Voice Memos (`.m4a`) can be decoded directly, with no conversion.
- Turn off noise suppression or "enhanced" recording modes if the app has them; they treat the watermark as noise.

**Computer (as Microphone)**
- [Audacity](https://www.audacityteam.org/) works well, or use the cell below to play and record in one step.

**Either way**
- Keep the volume moderate. A small speaker driven hard distorts, especially on bass.
- Do not use Bluetooth speakers or headsets; they compress the audio.

The cell below is optional. It plays the encoded file on this computer's speaker and records its microphone. It first plays a 3 s test tone to check that playback and recording are clean; only trust a measurement if that check passes.

In [ ]:
# OPTION: record with this computer (skip this cell if you record with a phone)
USE_LAPTOP = False            # <-- Set to True to play and record with this computer
SPEAKER    = None             # <-- None = default speaker, or part of its name
MICROPHONE = None             # <-- None = default microphone, or part of its name, e.g. "RODE"

if USE_LAPTOP:
    list_audio_devices()
    if check_audio_loop(SPEAKER, MICROPHONE):
        play_and_record(OUTPUT_AUDIO_FILE, "CrabRave_Recorded.wav", SPEAKER, MICROPHONE)

#### 1.6 Decoding the Recording

The decoder does not need the original music. The synchronisation pattern is part of the watermark itself, so it finds the message by searching the recording, and it also corrects for the speaker and microphone running at slightly different clock speeds. It does need the settings (level and message length), which are read from `CrabRave_Encoded.wm.json`, written next to the encoded file.

In [ ]:
# INPUT
RECORDED_AUDIO_FILE = "CrabRave_Recorded.wav"     # <-- Change this. Your recording (.wav, .m4a, .mp3)

SIMULATED = not os.path.exists(RECORDED_AUDIO_FILE)
if SIMULATED:
    SIMULATED_ROOM, SIMULATED_CM = "50cm", 50
    source = simulate_channel(OUTPUT_AUDIO_FILE, f"SIMULATED_{SIMULATED_ROOM}.wav", SIMULATED_ROOM)
    print("No recording found: decoding a SIMULATED 50 cm recording instead. Do not submit these results.\n")
else:
    source = RECORDED_AUDIO_FILE

got = wm.decode(source, settings=OUTPUT_AUDIO_FILE)
print(got)
print()
print(got.evaluate(MESSAGE))

#### 1.7 Logging Your Measurements

Every time you decode a recording, add it to a table. Keep the file: it is the raw data your results are built on. Simulated decodes go to a separate file, so `measurements.csv` only ever holds real recordings.

In [ ]:
# INPUT
DISTANCE_CM = 10                        # <-- Change this. Measured speaker-to-microphone distance
CSV_FILE    = "measurements.csv"

if SIMULATED:
    row = log_measurement("SIMULATED_measurements.csv", SIMULATED_CM, got, MESSAGE, note="SIMULATED")
    print(f"Simulated: logged to SIMULATED_measurements.csv, not to {CSV_FILE}.")
else:
    row = log_measurement(CSV_FILE, DISTANCE_CM, got, MESSAGE)
print(row)

In [ ]:
# Plot your real measurements so far: one line per level. Simulated rows are never plotted.
import pandas as pd
import matplotlib.pyplot as plt

has_rows = os.path.exists(CSV_FILE) and os.path.getsize(CSV_FILE) > 0
df = pd.read_csv(CSV_FILE) if has_rows else pd.DataFrame(columns=["note"])
df = df[df["note"].fillna("") != "SIMULATED"]
if df.empty:
    print(f"No real measurements in {CSV_FILE} yet: record, decode and log one first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for level, g in df.groupby("level"):
        g = g.groupby("distance_cm")[["ber", "char_accuracy"]].mean().reset_index()
        axes[0].plot(g["distance_cm"], g["ber"], "o-", label=f"level {level}")
        axes[1].plot(g["distance_cm"], g["char_accuracy"] * 100, "o-", label=f"level {level}")
    axes[0].set(xlabel="distance (cm)", ylabel="bit error rate", ylim=(-0.02, 0.6))
    axes[0].axhline(0.5, color="grey", ls="--", lw=1)
    axes[1].set(xlabel="distance (cm)", ylabel="character accuracy (%)", ylim=(-2, 102))
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()

#### 1.8 Exploring Without a Speaker

A simulated room lets you try every level quickly before you measure. Each level below is filled to its maximum message, because that is where the levels differ. The noise in the model was calibrated against one real speaker and microphone at 10 cm and 50 cm; your hardware will behave differently, so use this to explore and your recordings for your results.

In [ ]:
import numpy as np, soundfile as sf

LEVELS_TO_TRY = [2, 3, 4, 5, 6]     # <-- Change this.
ROOMS         = list(SCENARIOS)     # 10cm, 25cm, 50cm, 1m, 2m, 4m

seconds = sf.info(INPUT_AUDIO_FILE).duration
alphabet = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789 ")
print("simulated bit error rate, each level filled to its maximum message")
print(f"{'level':<7}{'chars':>6}" + "".join(f"{r:>8}" for r in ROOMS))
for level in LEVELS_TO_TRY:
    w = AudioWatermark(level=level, strength=STRENGTH)
    n = w.capacity(INPUT_AUDIO_FILE)["chars"]
    msg = "".join(np.random.default_rng(level).choice(alphabet, n))
    info = w.embed(INPUT_AUDIO_FILE, f"sim_level{level}.wav", msg)
    bers = []
    for room in ROOMS:
        simulate_channel(f"sim_level{level}.wav", "sim_room.wav", room)
        bers.append(w.decode("sim_room.wav", settings=info).evaluate(msg).ber)
    print(f"{level:<7}{n:>6}" + "".join(f"{b:>8.3f}" for b in bers))